# Tone-gap detection — validation notebook

Executed by Remyx on each validation and committed to this PR. It is the
**renderer**, not the experiment: it runs the repo's own
`eval/eval_tone_gap_detection.py` measurement and displays the result.


In [1]:
variant = 'baseline'
ref = 'main'
seed = 0


In [2]:
# Parameters
variant = "feature"
ref = "6de66588e2e2527cb0e7c7df7d586f8c3f2e8d79"
seed = 0


In [3]:
import asyncio, json, sys
from pathlib import Path

# papermill runs with --cwd set to the repo checkout, so cwd is the repo root.
# (The .py version uses __file__, which a notebook does not have.)
REPO_ROOT = Path.cwd()
for p in (REPO_ROOT, REPO_ROOT / 'eval'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print(f'variant={variant} ref={ref} seed={seed}')
print(f'repo root: {REPO_ROOT}')


variant=feature ref=6de66588e2e2527cb0e7c7df7d586f8c3f2e8d79 seed=0
repo root: /workspace/target_repo


## Measurement

The repo's own eval module, imported rather than reimplemented.


In [4]:
metrics = {
    'tone_sensitivity_verdict_accuracy': 0.0,
    'experience_metric_registry_count': 0.0,
    'verdicts_correct': 0.0,
    'conversations_total': 0.0,
}
try:
    import eval_tone_gap_detection as harness
    # ipykernel already runs an event loop, so asyncio.run() raises
    # 'cannot be called from a running event loop'. Top-level await is
    # supported here, and is what a notebook is supposed to use.
    metrics = await harness._measure()
    print('measured via the repo harness')
except Exception as exc:
    # Same degradation the .py version uses: emit comparable zeros rather
    # than crashing, so the baseline arm still produces a metrics line.
    # experience_metric_registry_count == 0 trips the spec guardrail, so a
    # genuinely broken run cannot masquerade as 'no change'.
    print(f'harness unavailable on this arm: {type(exc).__name__}: {exc}')


2026-09-07 23:32:50.196 | INFO     | pipecat:<module>:54 - ᓚᘏᗢ Pipecat 1.8.1 (Python 3.11.16 (main, Sep  1 2026, 00:12:26) [GCC 14.2.0]) ᓚᘏᗢ


measured via the repo harness


## Result


In [5]:
for k in sorted(metrics):
    print(f'{k:42} {metrics[k]}')


conversations_total                        15.0
experience_metric_registry_count           4.0
tone_sensitivity_verdict_accuracy          1.0
verdicts_correct                           15.0


In [6]:
# Provenance, rendered into the committed notebook so a reader can see
# which arm and commit produced these numbers.
print(f'arm={variant}  commit={ref}  seed={seed}')


arm=feature  commit=6de66588e2e2527cb0e7c7df7d586f8c3f2e8d79  seed=0


In [7]:
# Final line: the JSON the Remyx runner reads out of this notebook.
print(json.dumps({k: float(v) for k, v in metrics.items()}))


{"tone_sensitivity_verdict_accuracy": 1.0, "experience_metric_registry_count": 4.0, "verdicts_correct": 15.0, "conversations_total": 15.0}
